# Задание 3. Кластеризация банковских транзакций

Нужно подготовить данные, применить PCA и выполнить кластеризацию k-means.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

## 1. Загружаем данные

In [ ]:
bank_transactions = pd.read_csv(
    "/kaggle/input/datasets/zvictor12/bank-transactions/bank_transactions.csv",
    index_col="TransactionID"
)

bank_transactions.head()

## 2. Первичный анализ

In [ ]:
print("Размер:", bank_transactions.shape)
display(bank_transactions.info())
display(bank_transactions.isna().sum())
display(bank_transactions.nunique())

In [ ]:
plt.figure(figsize=(5, 4))
bank_transactions["CustGender"].value_counts(dropna=False).plot(kind="bar")
plt.title("CustGender")
plt.show()

plt.figure(figsize=(12, 5))
bank_transactions["CustLocation"].value_counts().head(20).plot(kind="bar")
plt.title("Top 20 CustLocation")
plt.xticks(rotation=60, ha="right")
plt.show()

In [ ]:
cols = ["CustAccountBalance", "TransactionAmount (INR)", "TransactionTime"]

bank_transactions[cols].hist(figsize=(12, 4), bins=50)
plt.tight_layout()
plt.show()

np.log1p(bank_transactions[["CustAccountBalance", "TransactionAmount (INR)"]]).hist(figsize=(10, 4), bins=50)
plt.tight_layout()
plt.show()

## 3. Подготовка признаков

Пропуски в балансе заполняем медианой. Для баланса и суммы транзакции используем `log1p`, потому что распределения сильно скошены вправо. Из времени берем час, из даты рождения считаем возраст. Редкие города объединяем в `OTHER`.

In [ ]:
bank_transactions["CustAccountBalance"] = bank_transactions["CustAccountBalance"].fillna(
    bank_transactions["CustAccountBalance"].median()
)

bank_transactions["CustGender"] = bank_transactions["CustGender"].fillna("U")
bank_transactions["CustGender"] = bank_transactions["CustGender"].where(
    bank_transactions["CustGender"].isin(["M", "F"]), "U"
)

top_locations = bank_transactions["CustLocation"].value_counts().head(10).index
bank_transactions["CustLocation"] = bank_transactions["CustLocation"].fillna("OTHER")
bank_transactions["CustLocation"] = bank_transactions["CustLocation"].where(
    bank_transactions["CustLocation"].isin(top_locations), "OTHER"
)

bank_transactions["balance_log"] = np.log1p(bank_transactions["CustAccountBalance"])
bank_transactions["amount_log"] = np.log1p(bank_transactions["TransactionAmount (INR)"])
bank_transactions["transaction_hour"] = (bank_transactions["TransactionTime"] // 10000).clip(0, 23)

bank_transactions["CustomerDOB_dt"] = pd.to_datetime(
    bank_transactions["CustomerDOB"],
    format="%d/%m/%y",
    errors="coerce"
)

bank_transactions["TransactionDate_dt"] = pd.to_datetime(
    bank_transactions["TransactionDate"],
    format="%d/%m/%y",
    errors="coerce"
)

mask = bank_transactions["CustomerDOB_dt"] > bank_transactions["TransactionDate_dt"]
bank_transactions.loc[mask, "CustomerDOB_dt"] = (
    bank_transactions.loc[mask, "CustomerDOB_dt"] - pd.DateOffset(years=100)
)

bank_transactions["age"] = (
    bank_transactions["TransactionDate_dt"] - bank_transactions["CustomerDOB_dt"]
).dt.days / 365.25

bank_transactions["age"] = bank_transactions["age"].where(
    bank_transactions["age"].between(10, 100),
    np.nan
)

bank_transactions["age"] = bank_transactions["age"].fillna(bank_transactions["age"].median())

bank_transactions[["CustGender", "CustLocation", "balance_log", "amount_log", "transaction_hour", "age"]].head()

In [ ]:
features = [
    "balance_log",
    "amount_log",
    "transaction_hour",
    "age",
    "CustGender",
    "CustLocation",
]

bank_features = bank_transactions[features]
bank_features = pd.get_dummies(bank_features, columns=["CustGender", "CustLocation"], dtype=int)

print(bank_features.shape)
bank_features.head()

## 4. Масштабирование

In [ ]:
scaler = StandardScaler()
bank_scaled = scaler.fit_transform(bank_features)

bank_scaled[:5]

## 5. PCA вручную

In [ ]:
def pca_manual(X, n_components=2):
    X_centered = X - X.mean(axis=0)
    cov_matrix = np.cov(X_centered, rowvar=False)

    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    components = eigenvectors[:, :n_components]
    X_pca = X_centered @ components
    explained_ratio = eigenvalues / eigenvalues.sum()

    return X_pca, components, explained_ratio

In [ ]:
bank_pca_2, bank_components_2, bank_explained_ratio = pca_manual(bank_scaled, n_components=2)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(bank_explained_ratio) + 1), np.cumsum(bank_explained_ratio), marker="o")
plt.xlabel("Количество компонент")
plt.ylabel("Накопленная доля дисперсии")
plt.grid()
plt.show()

print(bank_explained_ratio[:10])

In [ ]:
N_COMPONENTS = 15
bank_pca, bank_components, bank_explained_ratio = pca_manual(bank_scaled, n_components=N_COMPONENTS)

print(bank_pca.shape)
print("Сохраненная дисперсия:", bank_explained_ratio[:N_COMPONENTS].sum())

## 6. K-means вручную

In [ ]:
def kmeans_manual(X, k, max_iter=100, random_state=42):
    rng = np.random.default_rng(random_state)
    idx = rng.choice(len(X), k, replace=False)
    centroids = X[idx]

    for _ in range(max_iter):
        distances = np.sqrt(((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2))
        labels = np.argmin(distances, axis=1)

        new_centroids = np.array([
            X[labels == i].mean(axis=0) if np.any(labels == i) else centroids[i]
            for i in range(k)
        ])

        if np.allclose(centroids, new_centroids):
            break

        centroids = new_centroids

    return labels, centroids


def inertia_manual(X, labels, centroids):
    return sum(((X[labels == i] - centroids[i]) ** 2).sum() for i in range(len(centroids)))

In [ ]:
sample_size = min(50_000, len(bank_pca))
sample_idx = np.random.default_rng(42).choice(len(bank_pca), size=sample_size, replace=False)
bank_sample = bank_pca[sample_idx]

inertias = []

for k in range(2, 11):
    labels_sample, centroids_sample = kmeans_manual(bank_sample, k)
    inertias.append(inertia_manual(bank_sample, labels_sample, centroids_sample))

plt.figure(figsize=(8, 4))
plt.plot(range(2, 11), inertias, marker="o")
plt.xlabel("Количество кластеров")
plt.ylabel("Inertia")
plt.grid()
plt.show()

In [ ]:
K = 5
bank_labels, bank_centroids = kmeans_manual(bank_pca, k=K)

bank_transactions["cluster"] = bank_labels
bank_transactions["cluster"].value_counts().sort_index()

## 7. Интерпретация кластеров

In [ ]:
bank_cluster_means = bank_transactions.groupby("cluster")[[
    "CustAccountBalance",
    "TransactionAmount (INR)",
    "balance_log",
    "amount_log",
    "transaction_hour",
    "age",
]].mean()

bank_cluster_means

In [ ]:
pd.crosstab(bank_transactions["cluster"], bank_transactions["CustGender"], normalize="index")

In [ ]:
pd.crosstab(bank_transactions["cluster"], bank_transactions["CustLocation"], normalize="index")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(bank_pca_2[:, 0], bank_pca_2[:, 1], c=bank_labels, cmap="tab10", s=2)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Кластеры банковских транзакций")
plt.colorbar(label="cluster")
plt.show()

## Вывод

Кластеризация разделяет транзакции по сочетанию финансовых признаков, пола клиента, города, возраста и времени операции. Так как категориальные признаки были закодированы one-hot, часть кластеров может быть сильно связана с конкретными городами или группами городов. Для интерпретации используются средние значения по кластерам и доли категориальных признаков внутри каждого кластера.